# GeneVISTA source-data validation
Read-only preflight before notebook 02. No source records or labels are modified.

Default: inspect the first 10,000 records in each of six ClinVar files; fully scan the three HPO files and ClinGen table for structural validity. A sample passing does not verify the complete ClinVar file or its gzip checksum. Set `CLINVAR_SCAN_LIMIT = None` for a full scan.

Missing-field counts are diagnostics, not deletion rules. Exact clinical labels, including likely categories and mixed assertions, remain in the source. This notebook does not measure model accuracy.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path(r"D:\GeneVISTA")
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project directory missing: {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.dataset_validation import validate_datasets
CLINVAR_SCAN_LIMIT = 10_000
print(f"Project: {PROJECT_ROOT}")


Project: D:\GeneVISTA


In [2]:
report = validate_datasets(PROJECT_ROOT, clinvar_limit=CLINVAR_SCAN_LIMIT)
report_path = PROJECT_ROOT / "reports" / "dataset_validation_01.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Report: {report_path}")


PASSED: data/raw/clinvar/variant_summary_2022-01.txt.gz
PASSED: data/raw/clinvar/variant_summary_2023-01.txt.gz
PASSED: data/raw/clinvar/variant_summary_2024-01.txt.gz
PASSED: data/raw/clinvar/variant_summary_2025-01.txt.gz
PASSED: data/raw/clinvar/variant_summary_2026-01.txt.gz
PASSED: data/raw/clinvar/variant_summary_2026-09.txt.gz
PASSED: data/raw/hpo/hp.obo
PASSED: data/raw/hpo/genes_to_phenotype.txt
PASSED: data/raw/hpo/phenotype.hpoa
PASSED: data/raw/clingen/gene_disease_validity.csv
Report: D:\GeneVISTA\reports\dataset_validation_01.json


In [3]:
for result in report["results"]:
    if result["status"] == "failed":
        print(result["file"], result["error"])
    else:
        print(result["file"], result["scope"], result.get("rows_checked", result.get("terms_checked")))
        missing = {k:v for k,v in result.get("missing_values_observed", {}).items() if v}
        if missing:
            print("  Missing/placeholder values (review before cleaning):", missing)
failures = [r for r in report["results"] if r["status"] == "failed"]
if failures:
    raise RuntimeError(f"Validation failed for {len(failures)} of {report['expected']} files. See saved report.")
print(f"{report['passed']}/{report['expected']} files passed the configured structural checks.")
if CLINVAR_SCAN_LIMIT is not None:
    print("ClinVar checks are sampled; full-file integrity and downstream cleaning remain to be checked.")


data/raw/clinvar/variant_summary_2022-01.txt.gz first up to 10000 records only 10000
  Missing/placeholder values (review before cleaning): {'HGNC_ID': 28, 'LastEvaluated': 60, 'PhenotypeIDS': 82, 'PhenotypeList': 24, 'Assembly': 448, 'Chromosome': 448, 'ReferenceAllele': 9998, 'AlternateAllele': 9998, 'RCVaccession': 24}
data/raw/clinvar/variant_summary_2023-01.txt.gz first up to 10000 records only 10000
  Missing/placeholder values (review before cleaning): {'GeneSymbol': 2, 'HGNC_ID': 30, 'LastEvaluated': 68, 'PhenotypeIDS': 58, 'PhenotypeList': 32, 'Assembly': 134, 'Chromosome': 134, 'ReferenceAllele': 9998, 'AlternateAllele': 9998, 'RCVaccession': 28}
data/raw/clinvar/variant_summary_2024-01.txt.gz first up to 10000 records only 10000
  Missing/placeholder values (review before cleaning): {'GeneSymbol': 2, 'HGNC_ID': 37, 'LastEvaluated': 75, 'PhenotypeIDS': 57, 'PhenotypeList': 31, 'Assembly': 116, 'Chromosome': 116, 'ReferenceAllele': 9998, 'AlternateAllele': 9998, 'RCVaccession'